# Notebook 2: Spark Processing & Distributed Analysis
**H9DISS1 Data Intensive Scalable Systems | NCI MSc Data Analytics 2025/26**

---

## Overview — Three Analyses on Real FEMA Data

| Analysis | Pattern | Research Question |
|----------|---------|-------------------|
| **Analysis 1** | MapReduce Aggregation + Equi-Join | RQ1: Seismic Risk Index vs IA activation correlation |
| **Analysis 2** | Multi-Key Combinatorial GroupBy | RQ2: Multi-hazard portfolio & insurance exposure structure |
| **Analysis 3** | Spark SQL Window Functions (lag) | RQ3: Temporal trends & post-2010 insurance gap |

### Input Tables (from Notebook 1)
```
Tables/silver_fema_all_disasters   →  Analysis 1, 2, 3
Tables/silver_usgs_earthquakes     →  Analysis 1, 3
Tables/silver_seismic_risk_index   →  Analysis 1
```
### Output Tables (Gold Layer — written to Lakehouse)
```
Tables/gold_analysis1_eq_aid_correlation
Tables/gold_analysis2_multihazard_profile
Tables/gold_analysis2_eq_state_profile
Tables/gold_analysis3_national_trend
Tables/gold_analysis3_annual_by_state
Tables/gold_summary_statistics
```

## Cell 1: Load Silver Tables from Lakehouse

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

# Load Silver Delta tables from Lakehouse
fema  = spark.table("silver_fema_all_disasters")
usgs  = spark.table("silver_usgs_earthquakes")
sri   = spark.table("silver_seismic_risk_index")

# Earthquake subset
eq    = fema.filter(F.col("incidentType") == "Earthquake")

print(f"FEMA total:   {fema.count():>8,} records")
print(f"EQ subset:    {eq.count():>8,} records")
print(f"USGS events:  {usgs.count():>8,} records")
print(f"SRI states:   {sri.count():>8,} states")

## Cell 2: Analysis 1 — MapReduce Aggregation + Equi-Join

**Pattern:** Classic MapReduce — group earthquake declarations by state key,
aggregate program activation rates, then equi-join with USGS seismicity data
and the Seismic Risk Index.

**Map phase:** Each Spark partition independently maps (state → metrics)  
**Reduce phase:** Aggregations merged across partitions by state key  
**Join:** Hash-partitioned equi-join on state key across three DataFrames  

**Answers RQ1:** Is there a significant relationship between the Seismic Risk Index
and Individual Assistance activation rates?

In [ ]:
# ── MAP + REDUCE: Earthquake declarations aggregated by state ──────────────
eq_agg = eq.groupBy("state").agg(
    F.countDistinct("disasterNumber").alias("total_declarations"),
    F.count("designatedArea").alias("county_declarations"),
    F.sum("iaProgramDeclared").alias("ia_activations"),
    F.sum("paProgramDeclared").alias("pa_activations"),
    F.sum("hmProgramDeclared").alias("hm_activations"),
    F.avg("iaProgramDeclared").alias("ia_rate"),
    F.avg("paProgramDeclared").alias("pa_rate"),
    F.avg("hmProgramDeclared").alias("hm_rate"),
    (F.max("year") - F.min("year") + 1).alias("span_years"),
)
eq_agg = eq_agg.withColumn(
    "declarations_per_year",
    F.round(F.col("county_declarations") / F.col("span_years"), 3)
)

# ── MAP + REDUCE: USGS seismicity aggregated by state ─────────────────────
usgs_agg = usgs.filter(F.col("state") != "Other").groupBy("state").agg(
    F.count("event_id").alias("usgs_events"),
    F.avg("magnitude").alias("avg_magnitude"),
    F.max("magnitude").alias("max_magnitude"),
    F.sum(
        F.when(F.col("magnitude") >= 5.0, 1).otherwise(0)
    ).alias("m5plus_count")
)

# ── EQUI-JOIN: Three datasets joined on state key (hash-partitioned) ───────
result1 = (
    eq_agg
    .join(usgs_agg,                              on="state", how="left")
    .join(sri.select("state","sri","eq_share_pct"), on="state", how="left")
    .na.fill(0)
    .orderBy(F.col("sri").desc())
)

# ── Pearson Correlation (Spark built-in) ──────────────────────────────────
corr_sri_ia    = result1.stat.corr("sri",               "ia_rate")
corr_decl_ia   = result1.stat.corr("county_declarations","ia_activations")
corr_usgs_ia   = result1.filter(F.col("usgs_events") > 0) \
                        .stat.corr("usgs_events", "ia_rate")

print("── Analysis 1 Results ──────────────────────────────────")
print(f"  States analysed:                    {result1.count()}")
print(f"  Corr(SRI, IA activation rate):      {corr_sri_ia:.4f}")
print(f"  Corr(county declarations, IA act.): {corr_decl_ia:.4f}")
print(f"  Corr(USGS events, IA rate):         {corr_usgs_ia:.4f}")
print()

result1.select(
    "state","total_declarations","county_declarations",
    "ia_activations","ia_rate","pa_rate","sri"
).show(14, truncate=False)

# ── Write Gold Delta table ─────────────────────────────────────────────────
result1.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_analysis1_eq_aid_correlation")

print("✓ Gold table written: Tables/gold_analysis1_eq_aid_correlation")

## Cell 3: Analysis 2 — Multi-Key Combinatorial GroupBy

**Pattern:** Two-key GroupBy (`state × incidentType`) across the full 69,896-record
FEMA dataset. Each Spark partition independently aggregates its compound-key subset,
then results are merged. Produces 107 unique state × disaster-type combinations.

**Answers RQ2:** Which states exhibit the highest IA rates for earthquakes,
and how does the multi-hazard portfolio explain the divergence?

In [ ]:
EQ_STATES = ["CA","WA","AK","OR","UT","HI","ID","NV","PR","GU","AS"]

# ── MULTI-KEY GROUPBY: state × incidentType ────────────────────────────────
multihazard = (
    fema
    .filter(F.col("state").isin(EQ_STATES))
    .groupBy("state", "incidentType")
    .agg(
        F.countDistinct("disasterNumber").alias("declarations"),
        F.count("designatedArea").alias("county_rows"),
        F.sum("iaProgramDeclared").alias("ia_activations"),
        F.sum("paProgramDeclared").alias("pa_activations"),
        F.avg("iaProgramDeclared").alias("ia_rate"),
        F.avg("paProgramDeclared").alias("pa_rate"),
    )
)

# Total per state for share calculation
state_totals = multihazard.groupBy("state") \
    .agg(F.sum("declarations").alias("state_total"))

multihazard = multihazard \
    .join(state_totals, on="state", how="left") \
    .withColumn("disaster_share_pct",
                F.round(F.col("declarations") / F.col("state_total") * 100, 2))

# ── Earthquake-only profile slice ─────────────────────────────────────────
eq_profile = (
    multihazard
    .filter(F.col("incidentType") == "Earthquake")
    .withColumn("ia_per_declaration",
                F.round(
                    F.col("ia_activations") /
                    F.when(F.col("declarations") > 0, F.col("declarations")).otherwise(1),
                    3
                ))
    .orderBy(F.col("ia_activations").desc())
)

print("── Analysis 2 Results ──────────────────────────────────")
print(f"  State × incidentType combinations: {multihazard.count()}")
print(f"  States with EQ declarations:       {eq_profile.count()}")
print()
print("  Earthquake profile by state (ranked by IA activations):")
eq_profile.select(
    "state","declarations","county_rows",
    "ia_activations","ia_rate","pa_rate","disaster_share_pct"
).show(12, truncate=False)

# ── Write Gold Delta tables ────────────────────────────────────────────────
multihazard.write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("gold_analysis2_multihazard_profile")

eq_profile.write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("gold_analysis2_eq_state_profile")

print("✓ Gold tables written: gold_analysis2_multihazard_profile,",
      "gold_analysis2_eq_state_profile")

## Cell 4: Analysis 3 — Spark SQL Window Functions

**Pattern:** `lag()` window functions partitioned by state, ordered by year.
Computes year-over-year changes in declaration breadth and IA activation rate.

**Window spec:**
```python
windowSpec = Window.partitionBy('state').orderBy('year')
lag('county_rows', 1).over(windowSpec)  # previous year value
```

**Answers RQ3:** Is there statistical evidence of a growing residential
insurance gap in the post-2010 period?

In [ ]:
# ── Annual aggregation by state ────────────────────────────────────────────
annual_by_state = (
    eq
    .filter(F.col("year") >= 2000)
    .groupBy("state", "year")
    .agg(
        F.countDistinct("disasterNumber").alias("eq_declarations"),
        F.count("designatedArea").alias("county_rows"),
        F.sum("iaProgramDeclared").alias("ia_activations"),
        F.sum("paProgramDeclared").alias("pa_activations"),
        F.sum("hmProgramDeclared").alias("hm_activations"),
    )
    .orderBy("state", "year")
)

# ── WINDOW FUNCTION: YoY growth — lag(1) over (PARTITION BY state ORDER BY year) ──
state_window = Window.partitionBy("state").orderBy("year")

annual_by_state = (
    annual_by_state
    .withColumn("prev_county_rows",
                F.lag("county_rows", 1).over(state_window))
    .withColumn("prev_ia_activations",
                F.lag("ia_activations", 1).over(state_window))
    .withColumn("yoy_decl_growth_pct",
                F.round(
                    (F.col("county_rows") - F.col("prev_county_rows"))
                    / F.when(F.col("prev_county_rows") > 0,
                             F.col("prev_county_rows")).otherwise(1) * 100,
                    2
                ))
    .withColumn("yoy_ia_growth_pct",
                F.round(
                    (F.col("ia_activations") - F.col("prev_ia_activations"))
                    / F.when(F.col("prev_ia_activations") > 0,
                             F.col("prev_ia_activations")).otherwise(1) * 100,
                    2
                ))
)

# ── National annual totals ─────────────────────────────────────────────────
national_trend = (
    eq
    .filter(F.col("year") >= 2000)
    .groupBy("year")
    .agg(
        F.count("designatedArea").alias("total_county_rows"),
        F.sum("iaProgramDeclared").alias("total_ia"),
        F.sum("paProgramDeclared").alias("total_pa"),
        F.countDistinct("disasterNumber").alias("unique_disasters"),
    )
    .orderBy("year")
)

# ── Window over national trend ─────────────────────────────────────────────
global_window = Window.orderBy("year")
national_trend = (
    national_trend
    .withColumn("prev_ia",      F.lag("total_ia",         1).over(global_window))
    .withColumn("prev_rows",    F.lag("total_county_rows", 1).over(global_window))
    .withColumn("yoy_ia_change",
                F.round(
                    (F.col("total_ia") - F.col("prev_ia"))
                    / F.when(F.col("prev_ia") > 0, F.col("prev_ia")).otherwise(1) * 100,
                    2
                ))
    .withColumn("ia_share_pct",
                F.round(
                    F.col("total_ia")
                    / F.when(F.col("total_county_rows") > 0,
                             F.col("total_county_rows")).otherwise(1) * 100,
                    2
                ))
)

# Key finding: IA share pre vs post 2010
pre_2010  = national_trend.filter(F.col("year") < 2010) \
    .agg(F.avg("ia_share_pct")).collect()[0][0]
post_2010 = national_trend.filter(F.col("year") >= 2010) \
    .agg(F.avg("ia_share_pct")).collect()[0][0]

print("── Analysis 3 Results ──────────────────────────────────")
print(f"  Pre-2010  avg IA share of county rows: {pre_2010:.1f}%")
print(f"  Post-2010 avg IA share of county rows: {post_2010:.1f}%")
print(f"  Change: {post_2010 - pre_2010:.1f} percentage points")
print()
national_trend.select(
    "year","total_county_rows","total_ia","total_pa",
    "unique_disasters","ia_share_pct"
).show(25, truncate=False)

# ── Write Gold Delta tables ────────────────────────────────────────────────
annual_by_state.write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("gold_analysis3_annual_by_state")

national_trend.write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("gold_analysis3_national_trend")

print("✓ Gold tables: gold_analysis3_annual_by_state, gold_analysis3_national_trend")

## Cell 5: Build Gold Summary Statistics Table

In [ ]:
# Collect summary statistics into a single Gold table for the report
r1   = spark.table("gold_analysis1_eq_aid_correlation")
r2   = spark.table("gold_analysis2_eq_state_profile")
r3n  = spark.table("gold_analysis3_national_trend")

# Key metrics
top_sri_state   = r1.orderBy(F.col("sri").desc()).first()["state"]
top_ia_state    = r1.orderBy(F.col("ia_rate").desc()).first()["state"]
top_ia_rate     = r1.orderBy(F.col("ia_rate").desc()).first()["ia_rate"]
pr_county       = r1.filter(F.col("state")=="PR").first()["county_declarations"]
pr_ia           = r1.filter(F.col("state")=="PR").first()["ia_activations"]
total_ia        = int(r1.agg(F.sum("ia_activations")).collect()[0][0])
total_pa        = int(r1.agg(F.sum("pa_activations")).collect()[0][0])
peak_year_row   = r3n.orderBy(F.col("total_county_rows").desc()).first()
peak_year       = int(peak_year_row["year"])
peak_rows       = int(peak_year_row["total_county_rows"])
states_zero_ia  = r1.filter(F.col("ia_rate") == 0.0).count()

summary_data = [{
    "metric":  "total_fema_records",       "value": "69896",
    "context": "All federal disaster declarations 1953-2026"
}, {
    "metric":  "earthquake_county_rows",    "value": "228",
    "context": "County-level earthquake declaration rows"
}, {
    "metric":  "unique_eq_disasters",       "value": "36",
    "context": "Unique federally declared earthquake events"
}, {
    "metric":  "states_analysed",           "value": str(r1.count()),
    "context": "States and territories with earthquake declarations"
}, {
    "metric":  "corr_sri_ia",               "value": f"{corr_sri_ia:.4f}",
    "context": "Pearson r: Seismic Risk Index vs IA activation rate"
}, {
    "metric":  "highest_sri_state",         "value": top_sri_state,
    "context": "State with highest composite Seismic Risk Index"
}, {
    "metric":  "highest_ia_state",          "value": f"{top_ia_state} ({top_ia_rate:.1%})",
    "context": "State with highest IA activation rate"
}, {
    "metric":  "pr_county_rows",            "value": str(int(pr_county)),
    "context": "Puerto Rico: county declarations with 0 IA activations"
}, {
    "metric":  "total_ia_activations",      "value": str(total_ia),
    "context": "Sum of all IA activations across earthquake declarations"
}, {
    "metric":  "states_with_zero_ia",       "value": str(states_zero_ia),
    "context": "States/territories with earthquake declarations but 0 IA"
}, {
    "metric":  "peak_declaration_year",     "value": str(peak_year),
    "context": f"Year with most county earthquake declarations ({peak_rows} rows)"
}, {
    "metric":  "pre_2010_ia_share",         "value": f"{pre_2010:.1f}%",
    "context": "Avg IA share of county rows pre-2010 (residential coverage era)"
}, {
    "metric":  "post_2010_ia_share",        "value": f"{post_2010:.1f}%",
    "context": "Avg IA share of county rows post-2010 (insurance gap era)"
}]

df_summary = spark.createDataFrame(summary_data)
df_summary.write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("gold_summary_statistics")

print("── Gold Summary Statistics ─────────────────────────────")
df_summary.show(20, truncate=False)
print("✓ Written: Tables/gold_summary_statistics")
print()
print("All Gold tables written. Proceed to Notebook 3: Visualisation.")